# Debug our own model

Diagnostics only -- no training here. Loads a checkpoint written by
`train_own_model.ipynb` and asks *why* val mAP is low, one head branch at a time:

- **threshold**: is mAP being truncated by the confidence cutoff?
- **class-agnostic mAP**: are boxes + objectness fine, with classification the only blocker?
- **`branch_stats`**: objectness / classification / box quality read directly at the
  ground-truth cells, on val *and* train (overfitting vs. not learning).
- **overfit test** (section 9): can a fresh model memorize 64 images, under each cls/obj
  loss normalization? Independent of sections 5-8 -- needs only sections 1-5 run first.

Reuses the exact split saved by `train_own_model.ipynb` (`manifest.json` on Drive),
so the numbers are comparable to the training log.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
runDir = f"{RUN_ROOT}/own_model"

## 2. Download the dataset

The saved manifest stores absolute image paths, so the images have to exist at the
same location as during training -- same Roboflow download, same `/content` cwd.

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

datasetDirectory = dataset.location
print(datasetDirectory)

## 3. Clone (or update) the repo

In [ ]:
import os
import sys

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt
!git -C {repoDirectory} log --oneline -1

sys.path.insert(0, repoDirectory)

## 4. Load the saved split

In [ ]:
import json

with open(f"{runDir}/manifest.json") as f:
    manifest = json.load(f)
with open(f"{runDir}/classes.json") as f:
    classNames = json.load(f)
numClasses = len(classNames)

for setName in ["train", "valid", "test"]:
    print(setName, ":", len(manifest[setName]), "images")

missing = [s["image_path"] for s in manifest["valid"] + manifest["train"] if not os.path.exists(s["image_path"])]
assert not missing, f"{len(missing)} manifest images not found, e.g. {missing[0]} -- dataset path differs from training"

## 5. Load the checkpoint

`latest.pt`, not `best.pt`: `best.pt` is only rewritten when val mAP improves, and
with the old 0.3 eval threshold mAP was stuck at 0.0, so `best.pt` may still be epoch 0.
These settings must match what `train_model.py` was run with.

In [ ]:
import torch
from torch.utils.data import DataLoader

from data.dataset import DetectionDataset, collate_fn
from models.yolo import YOLO
from train.trainer import load_checkpoint, evaluate_map
from train.diagnostics import branch_stats

device = "cuda" if torch.cuda.is_available() else "cpu"
imgSize = 320
strides = (8, 16, 32)
thresholds = (32, 80)
checkpointPath = f"{runDir}/latest.pt"

model = YOLO(numClasses).to(device)
state = load_checkpoint(checkpointPath, model, device=device)
model.eval()
print("checkpoint:", checkpointPath, "| epoch:", state["epoch"], "| device:", device)

valLoader = DataLoader(DetectionDataset(manifest["valid"], img_size=imgSize),
                       batch_size=16, shuffle=False, collate_fn=collate_fn)
# a fixed train subset, enough to compare against val without a full pass
trainLoader = DataLoader(DetectionDataset(manifest["train"][:1000], img_size=imgSize),
                         batch_size=16, shuffle=False, collate_fn=collate_fn)

## 6. mAP vs. confidence threshold, normal and class-agnostic

- mAP rising as the threshold drops -> the cutoff was hiding true positives.
- class-agnostic >> normal -> boxes/objectness work, classification is what's costing mAP.
- class-agnostic also ~0 -> look at the `box_iou` columns below.

In [ ]:
import pandas as pd

rows = []
for t in (0.3, 0.01, 0.001):
    rows.append({
        "conf_thresh": t,
        "mAP@0.5": evaluate_map(model, valLoader, device, strides, numClasses, conf_thresh=t)[0.5],
        "class-agnostic mAP@0.5": evaluate_map(model, valLoader, device, strides, numClasses,
                                               conf_thresh=t, class_agnostic=True)[0.5],
    })
pd.DataFrame(rows).set_index("conf_thresh").round(4)

## 7. Per-branch stats, val vs. train

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

statsByName = {
    "val": branch_stats(model, valLoader, device, imgSize, strides, numClasses, thresholds),
    "train[:1000]": branch_stats(model, trainLoader, device, imgSize, strides, numClasses, thresholds),
}

table = pd.DataFrame({
    (name, level): s for name, stats in statsByName.items() for level, s in stats.items()
}).T
table.round(4)

## 8. The questions that pick the fix

Read off the `all` rows:

| check | compare | meaning |
|---|---|---|
| is the classifier just guessing the most common class? | `cls_top1_acc` vs `majority_share`, plus `pred_mode_share` | top-1 ~ majority share and a high mode share = collapse onto one class |
| overfitting or not learning? | train `cls_top1_acc` vs val `cls_top1_acc` | big gap -> overfitting (augment, no h-flips); both low -> too little signal (center sampling) |
| are boxes OK? | `box_iou_median`, `box_iou>=0.5` | low -> localization is a problem too, independent of class |
| is objectness OK? | `neg_above_pos_median` | ~0 = ranks objects above background; values may still be low (`obj_pos_median`) |

In [ ]:
for name, stats in statsByName.items():
    s = stats["all"]
    print(f"--- {name} ---")
    print(f"top-1 acc            {s['cls_top1_acc']:.3f}   (chance {1/numClasses:.3f})")
    print(f"majority-class share {s['majority_share']:.3f}   class: {classNames[s['majority_class']]}")
    print(f"most-predicted share {s['pred_mode_share']:.3f}   class: {classNames[s['pred_mode_class']]}"
          f"   ({s['n_distinct_pred']} distinct classes predicted)")
    print(f"box IoU median       {s['box_iou_median']:.3f}   (>=0.5 for {s['box_iou>=0.5']:.1%} of objects)")
    print(f"obj: pos median {s['obj_pos_median']:.3f} vs neg mean {s['obj_neg_mean']:.4f},"
          f" negatives above pos median {s['neg_above_pos_median']:.2%}")
    print()

## 9. Overfit test: can the detector memorize 64 images?

A fresh model, trained from scratch on just 64 training images, then scored on those
**same** images. A healthy detector should memorize them: cls top-1 near 1.0, and
mAP near 1.0. Each run below is one loss configuration (see `train/loss.py`); every
run starts from the same seed, so the config is the only difference.

This cell only defines the helpers -- sections 9a/9b run the actual comparisons and
are independent of each other (run either, or both).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from train.targets import build_targets_batch
from train.loss import yolo_loss

overfitBatches = list(DataLoader(DetectionDataset(manifest["train"][:64], img_size=imgSize),
                                 batch_size=16, shuffle=False, collate_fn=collate_fn))
overfitEpochs = 150
evalEvery = 10
trackedStats = ["cls_top1_acc", "majority_share", "pred_mode_share", "n_distinct_pred",
                "box_iou_median", "obj_pos_median", "obj_neg_mean", "neg_above_pos_median"]


def overfit(**lossKwargs):
    """lossKwargs go straight to yolo_loss, e.g. cls_norm="positive", obj_norm="balanced"."""
    torch.manual_seed(0)  # same init for every run -- the loss config is the only difference
    net = YOLO(numClasses).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    tag = ", ".join(f"{k}={v}" for k, v in lossKwargs.items())
    history = []

    for epoch in range(1, overfitEpochs + 1):
        net.train()
        for images, boxesList, labelsList in overfitBatches:
            preds = net(images.to(device))
            targets = build_targets_batch(boxesList, labelsList, imgSize, imgSize, strides,
                                          numClasses, thresholds)
            targets = [tuple(t.to(device) for t in level) for level in targets]
            losses = yolo_loss(preds, targets, **lossKwargs)

            optimizer.zero_grad()
            losses["total"].backward()
            optimizer.step()

        if epoch % evalEvery == 0:
            s = branch_stats(net, overfitBatches, device, imgSize, strides, numClasses, thresholds)["all"]
            mAP = evaluate_map(net, overfitBatches, device, strides, numClasses)[0.5]
            history.append({"epoch": epoch, "mAP@0.5": mAP, **{k: s[k] for k in trackedStats}})
            print(f"[{tag}] epoch {epoch}: mAP {mAP:.3f}, top-1 {s['cls_top1_acc']:.3f}, "
                  f"obj pos median {s['obj_pos_median']:.3f}, neg above pos {s['neg_above_pos_median']:.2%}")

    return net, pd.DataFrame(history).set_index("epoch")


def compare(results, columns, baselines=None):
    """results: {label: (net, history)}. One subplot per column, one line per run,
    then a final-epoch table (plus class-agnostic mAP)."""
    figure, axes = plt.subplots(1, len(columns), figsize=(6.5 * len(columns), 4), squeeze=False)
    for axis, column in zip(axes[0], columns):
        for label, (_, history) in results.items():
            axis.plot(history.index, history[column], label=label)
        if baselines and column in baselines:
            value, text = baselines[column]
            axis.axhline(value, color="gray", linestyle="--", label=text)
        axis.set_title(column)
        axis.set_xlabel("epoch")
        axis.set_ylim(0, 1.05)
        axis.legend()
    plt.tight_layout()
    plt.show()

    rows = []
    for label, (net, history) in results.items():
        rows.append({
            "run": label,
            **history.iloc[-1].to_dict(),
            "class-agnostic mAP@0.5": evaluate_map(net, overfitBatches, device, strides, numClasses,
                                                   class_agnostic=True)[0.5],
        })
    return pd.DataFrame(rows).set_index("run").round(4)

### 9a. Classification loss normalization -- already run, result: `positive` wins

- `element` -- the original: cls BCE averaged over positive cells x 54 classes
- `positive` -- YOLOv8's: summed over classes, averaged over positive cells

Measured (2026-10-22): `positive` reached top-1 1.0 by ~epoch 50, `element` crawled to
0.30 with the majority-class collapse. Kept here so the result is reproducible; skip it
to save time. Both runs use `obj_norm="element"` -- the loss as it was at the time.

In [ ]:
clsResults = {
    f"cls_norm={c}": overfit(cls_norm=c, obj_norm="element") for c in ("element", "positive")
}
majority = next(iter(clsResults.values()))[1]["majority_share"].iloc[0]
compare(clsResults, ["cls_top1_acc", "pred_mode_share", "mAP@0.5"],
        baselines={"cls_top1_acc": (majority, "majority-class baseline")})

### 9b. Objectness loss normalization

With the cls fix in, 9a's `positive` run still only reached mAP 0.25 on images it had
memorized -- objectness at true object cells had collapsed to a median of ~0.001.

- `element` -- the original: one weighted BCE mean over all ~2,100 cells per image
- `balanced` -- positive and negative cells each averaged on their own, then weighted

Both runs use `cls_norm="positive"`. What to look at:

- `obj_pos_median` -- should climb toward 1 under `balanced`
- `neg_above_pos_median` -- fraction of empty cells outranking a typical real object;
  should stay ~0. If it *rises* under `balanced`, positives are now over-weighted and
  empty cells near objects are lighting up -> try a larger `noobj_weight` (third run below)
- `mAP@0.5` -- the actual goal: near 1.0 on memorized images

In [ ]:
objResults = {
    "obj_norm=element": overfit(cls_norm="positive", obj_norm="element"),
    "obj_norm=balanced": overfit(cls_norm="positive", obj_norm="balanced"),
    # uncomment if balanced shows false positives (neg_above_pos_median rising):
    # "obj_norm=balanced, noobj=2.0": overfit(cls_norm="positive", obj_norm="balanced", noobj_weight=2.0),
}
compare(objResults, ["mAP@0.5", "obj_pos_median", "neg_above_pos_median"])